# training deep learning models on c4 only 
- using the matched dataset made in crumb.ipynb


# 1. imports and that

In [ ]:
# Deep Learning Models for Autism Diagnosis Prediction
# Using Age-Matched Dataset with Comprehensive Feature Engineering

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. load data and basic FE

In [ ]:
# Load and prepare the age-matched dataset
print("=== LOADING AGE-MATCHED DATASET ===")
df_age_matched = pd.read_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_strict_age_matched_balanced.csv')

print(f"Dataset shape: {df_age_matched.shape}")
print(f"Target distribution: {df_age_matched['autism_target'].value_counts()}")

# Feature Engineering - Part 1 (Existing Features)
def create_aggregate_features(df):
    """Create aggregate features from questionnaire items"""
    # AQ subdomain scores
    df['aq_social_skills'] = df[['aq_1', 'aq_2', 'aq_4']].sum(axis=1)
    df['aq_attention_switching'] = df[['aq_3', 'aq_5', 'aq_6']].sum(axis=1)
    df['aq_attention_to_detail'] = df[['aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
    
    # EQ subdomain scores
    df['eq_cognitive'] = df[['eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5']].sum(axis=1)
    df['eq_affective'] = df[['eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10']].sum(axis=1)
    
    # SQR subdomain scores
    df['sqr_social_awareness'] = df[['sqr_1', 'sqr_2']].sum(axis=1)
    df['sqr_social_cognition'] = df[['sqr_3', 'sqr_4', 'sqr_5']].sum(axis=1)
    df['sqr_social_communication'] = df[['sqr_6', 'sqr_7', 'sqr_8']].sum(axis=1)
    df['sqr_social_motivation'] = df[['sqr_9', 'sqr_10']].sum(axis=1)
    
    # SPQ subdomain scores
    df['spq_cognitive_perceptual'] = df[['spq_1', 'spq_2', 'spq_3', 'spq_4']].sum(axis=1)
    df['spq_interpersonal'] = df[['spq_5', 'spq_6', 'spq_7', 'spq_8']].sum(axis=1)
    df['spq_disorganized'] = df[['spq_9', 'spq_10']].sum(axis=1)
    
    return df

# Apply feature engineering
df_age_matched = create_aggregate_features(df_age_matched)

# 3. advanced FE

In [ ]:
# Feature Engineering - Part 2 (Scientific Literature-Based Features)
print("=== APPLYING FEATURE ENGINEERING ===")

# Age-related features
df_age_matched['age_squared'] = df_age_matched['age'] ** 2
df_age_matched['age_cubed'] = df_age_matched['age'] ** 3
df_age_matched['age_log'] = np.log1p(df_age_matched['age'])

# Age groups
df_age_matched['age_group'] = pd.cut(df_age_matched['age'], 
                                    bins=[0, 18, 25, 35, 50, 100], 
                                    labels=['0-18', '19-25', '26-35', '36-50', '50+'])

# Sex interactions
if 'sex_num' in df_age_matched.columns:
    df_age_matched['sex_x_aq'] = df_age_matched['sex_num'] * df_age_matched['aq_total']
    df_age_matched['sex_x_eq'] = df_age_matched['sex_num'] * df_age_matched['eq_total']
    df_age_matched['sex_x_sqr'] = df_age_matched['sex_num'] * df_age_matched['sqr_total']
    df_age_matched['sex_x_spq'] = df_age_matched['sex_num'] * df_age_matched['spq_total']

# STEM interactions
if 'is_stem_occupation' in df_age_matched.columns:
    df_age_matched['stem_x_aq'] = df_age_matched['is_stem_occupation'] * df_age_matched['aq_total']
    df_age_matched['stem_x_eq'] = df_age_matched['is_stem_occupation'] * df_age_matched['eq_total']

# Feature Engineering - Part 3 (Statistical Features)
# D-score and interactions
df_age_matched['d_score'] = df_age_matched['eq_total'] - df_age_matched['sqr_total']
df_age_matched['age_x_aq'] = df_age_matched['age'] * df_age_matched['aq_total']
df_age_matched['age_x_eq'] = df_age_matched['age'] * df_age_matched['eq_total']
df_age_matched['aq_eq_interaction'] = df_age_matched['aq_total'] * df_age_matched['eq_total']

# Ratios
df_age_matched['eq_sqr_ratio'] = df_age_matched['eq_total'] / (df_age_matched['sqr_total'] + 1e-8)
df_age_matched['aq_eq_ratio'] = df_age_matched['aq_total'] / (df_age_matched['eq_total'] + 1e-8)
df_age_matched['aq_spq_ratio'] = df_age_matched['aq_total'] / (df_age_matched['spq_total'] + 1e-8)

# Transformations
df_age_matched['log_aq_total'] = np.log1p(df_age_matched['aq_total'])
df_age_matched['sqrt_age'] = np.sqrt(df_age_matched['age'])

# High/low trait flags
aq_mean = df_age_matched['aq_total'].mean()
aq_std = df_age_matched['aq_total'].std()
df_age_matched['high_aq'] = (df_age_matched['aq_total'] > aq_mean + aq_std).astype(int)

eq_mean = df_age_matched['eq_total'].mean()
eq_std = df_age_matched['eq_total'].std()
df_age_matched['low_eq'] = (df_age_matched['eq_total'] < eq_mean - eq_std).astype(int)

# Z-scores and percentile ranks
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df_age_matched.columns:
        df_age_matched[f'{col}_zscore'] = (df_age_matched[col] - df_age_matched[col].mean()) / df_age_matched[col].std()
        df_age_matched[f'{col}_percentile'] = df_age_matched[col].rank(pct=True)

# Extreme value indicators
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df_age_matched.columns:
        q95 = df_age_matched[col].quantile(0.95)
        q05 = df_age_matched[col].quantile(0.05)
        df_age_matched[f'{col}_high_extreme'] = (df_age_matched[col] > q95).astype(int)
        df_age_matched[f'{col}_low_extreme'] = (df_age_matched[col] < q05).astype(int)

# Three-way interactions
df_age_matched['age_sex_aq'] = df_age_matched['age'] * df_age_matched['sex_num'] * df_age_matched['aq_total']
df_age_matched['age_sex_eq'] = df_age_matched['age'] * df_age_matched['sex_num'] * df_age_matched['eq_total']

# Quadratic terms
df_age_matched['aq_total_squared'] = df_age_matched['aq_total'] ** 2
df_age_matched['eq_total_squared'] = df_age_matched['eq_total'] ** 2
df_age_matched['sqr_total_squared'] = df_age_matched['sqr_total'] ** 2

# Cross-ratios
df_age_matched['aq_eq_cross_ratio'] = df_age_matched['aq_total'] * df_age_matched['eq_total'] / (df_age_matched['sqr_total'] + 1e-8)
df_age_matched['aq_sqr_cross_ratio'] = df_age_matched['aq_total'] * df_age_matched['sqr_total'] / (df_age_matched['eq_total'] + 1e-8)
df_age_matched['eq_sqr_cross_ratio'] = df_age_matched['eq_total'] * df_age_matched['sqr_total'] / (df_age_matched['aq_total'] + 1e-8)

print(f"After feature engineering: {df_age_matched.shape}")

# 4. feature selection adn data prep

In [ ]:
# Feature Selection and Data Preparation
print("=== FEATURE SELECTION AND DATA PREPARATION ===")

# Prepare target and features
y = df_age_matched['autism_target']
X = df_age_matched.drop(['autism_target'], axis=1)

# Remove any features containing 'autism', 'risk_score', or 'target'
leakage_features = [col for col in X.columns if any(term in col.lower() for term in ['autism', 'risk_score', 'target'])]
X = X.drop(columns=leakage_features)

print(f"Removed leakage features: {leakage_features}")

# Remove non-numeric and constant features
X = X.select_dtypes(include=[np.number])
constant_features = X.columns[X.std() == 0]
X = X.drop(columns=constant_features)

print(f"Removed constant features: {len(constant_features)}")

# Handle missing values
missing_counts = X.isnull().sum()
if missing_counts.sum() > 0:
    print(f"Missing values found: {missing_counts.sum()}")
    X = X.fillna(X.mean())
else:
    print("No missing values found")

print(f"Final feature set: {X.shape[1]} features")
print("Sample features:", list(X.columns[:10]))

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Target distribution - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")

# 5. data prep fucntions and model definitions 

In [ ]:
# Data preparation for deep learning
def prepare_data_for_dl(X_train, X_test, y_train, y_test, batch_size=512):
    """Prepare data for deep learning models"""
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train)
    X_test_tensor = torch.FloatTensor(X_test)
    y_train_tensor = torch.LongTensor(y_train.values)
    y_test_tensor = torch.LongTensor(y_test.values)
    
    # Create datasets
    train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader, X_train.shape[1]

# Deep Learning Models

# Model 1: Simple Feedforward Neural Network
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, dropout=0.3):
        super(SimpleNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 2)
        )
    
    def forward(self, x):
        return self.layers(x)

# Model 2: Residual Network
class ResidualBlock(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(ResidualBlock, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, input_size)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        residual = x
        out = self.relu(self.linear1(x))
        out = self.linear2(out)
        out += residual
        return self.relu(out)

class ResidualNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_blocks=3):
        super(ResidualNN, self).__init__()
        self.input_layer = nn.Linear(input_size, hidden_size)
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(hidden_size, hidden_size) for _ in range(num_blocks)
        ])
        self.output_layer = nn.Linear(hidden_size, 2)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.input_layer(x)
        for block in self.residual_blocks:
            x = block(x)
            x = self.dropout(x)
        return self.output_layer(x)

# Model 3: Attention-based Network
class AttentionLayer(nn.Module):
    def __init__(self, input_size, attention_size=32):
        super(AttentionLayer, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_size, attention_size),
            nn.Tanh(),
            nn.Linear(attention_size, 1),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        attention_weights = self.attention(x)
        attended = torch.sum(attention_weights * x, dim=1)
        return attended

class AttentionNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, attention_size=32):
        super(AttentionNN, self).__init__()
        self.feature_embedding = nn.Linear(input_size, hidden_size)
        self.attention = AttentionLayer(hidden_size, attention_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size // 2, 2)
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x = self.feature_embedding(x)
        x = self.attention(x)
        return self.classifier(x)

# Model 4: Convolutional Neural Network (1D)
class CNN1D(nn.Module):
    def __init__(self, input_size, num_filters=32, kernel_size=3):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(1, num_filters, kernel_size, padding=kernel_size//2)
        self.conv2 = nn.Conv1d(num_filters, num_filters*2, kernel_size, padding=kernel_size//2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(num_filters*2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        return self.classifier(x)

# Model 5: Ensemble of Small Networks
class EnsembleNN(nn.Module):
    def __init__(self, input_size, num_models=3):
        super(EnsembleNN, self).__init__()
        self.models = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_size, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(32, 2)
            ) for _ in range(num_models)
        ])
    
    def forward(self, x):
        outputs = [model(x) for model in self.models]
        return torch.stack(outputs).mean(dim=0)

# 6. training and evaluation functions

In [ ]:
# Training function
def train_model(model, train_loader, test_loader, epochs=20, lr=0.001, patience=5):
    """Train a PyTorch model with early stopping"""
    
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
                preds = torch.softmax(outputs, dim=1)
                all_preds.extend(preds[:, 1].cpu().numpy())
                all_labels.extend(batch_y.cpu().numpy())
        
        train_loss /= len(train_loader)
        val_loss /= len(test_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
        
        if epoch % 5 == 0:
            auc = roc_auc_score(all_labels, all_preds)
            print(f"Epoch {epoch}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, AUC: {auc:.4f}")
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    return model, train_losses, val_losses

# Evaluation function
def evaluate_model(model, test_loader, y_test):
    """Evaluate model performance"""
    model.eval()
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    
    accuracy = accuracy_score(y_test, all_preds)
    auc = roc_auc_score(y_test, all_probs)
    f1 = f1_score(y_test, all_preds)
    
    return accuracy, auc, f1, all_preds, all_probs

# 7. main execution function to run DL

In [ ]:
# Main execution
def run_deep_learning_experiments(X_train_scaled, X_test_scaled, y_train, y_test):
    """Run all deep learning experiments"""
    
    print("Preparing data for deep learning...")
    train_loader, test_loader, input_size = prepare_data_for_dl(
        X_train_scaled, X_test_scaled, y_train, y_test
    )
    
    models = {
        'SimpleNN': SimpleNN(input_size),
        'ResidualNN': ResidualNN(input_size),
        'AttentionNN': AttentionNN(input_size),
        'CNN1D': CNN1D(input_size),
        'EnsembleNN': EnsembleNN(input_size)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n{'='*50}")
        print(f"Training {name}")
        print(f"{'='*50}")
        
        try:
            # Train model
            trained_model, train_losses, val_losses = train_model(
                model, train_loader, test_loader, epochs=25
            )
            
            # Evaluate
            accuracy, auc, f1, preds, probs = evaluate_model(
                trained_model, test_loader, y_test
            )
            
            results[name] = {
                'accuracy': accuracy,
                'auc': auc,
                'f1': f1,
                'model': trained_model,
                'train_losses': train_losses,
                'val_losses': val_losses
            }
            
            print(f"\n{name} Results:")
            print(f"Accuracy: {accuracy:.4f}")
            print(f"AUC: {auc:.4f}")
            print(f"F1 Score: {f1:.4f}")
            
            # Plot training curves
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(train_losses, label='Train Loss')
            plt.plot(val_losses, label='Val Loss')
            plt.title(f'{name} - Training Curves')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            
            plt.subplot(1, 2, 2)
            plt.hist(probs, bins=50, alpha=0.7)
            plt.title(f'{name} - Prediction Probabilities')
            plt.xlabel('Probability')
            plt.ylabel('Count')
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error training {name}: {e}")
            results[name] = {'error': str(e)}
    
    return results

# Run the experiments
print("=== RUNNING DEEP LEARNING EXPERIMENTS ===")
dl_results = run_deep_learning_experiments(X_train_scaled, X_test_scaled, y_train, y_test)

# 8. results summary and comparison

In [ ]:
# Compare with traditional ML results
print("\n" + "="*60)
print("DEEP LEARNING vs TRADITIONAL ML COMPARISON")
print("="*60)

comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Type': 'Deep Learning',
        'Accuracy': result.get('accuracy', np.nan),
        'AUC': result.get('auc', np.nan),
        'F1': result.get('f1', np.nan)
    }
    for name, result in dl_results.items() if 'error' not in result
]).sort_values('AUC', ascending=False)

print(comparison_df.to_string(index=False))

# Find best model
if len(comparison_df) > 0:
    best_model_name = comparison_df.iloc[0]['Model']
    best_model = dl_results[best_model_name]['model']
    
    print(f"\nBEST DEEP LEARNING MODEL: {best_model_name}")
    print(f"Best AUC: {comparison_df.iloc[0]['AUC']:.4f}")
    print(f"Best Accuracy: {comparison_df.iloc[0]['Accuracy']:.4f}")
    
    # Detailed analysis of best model
    print(f"\nDetailed analysis of {best_model_name}...")
    
    # Confusion matrix
    _, _, _, preds, probs = evaluate_model(best_model, test_loader, y_test)
    
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {best_model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, preds))

print("\n" + "="*60)
print("FEATURE ENGINEERING SUMMARY")
print("="*60)
print(f"Original features: 68")
print(f"Engineered features: {X.shape[1]}")
print(f"Total features: {X.shape[1]}")
print("="*60)

# tabnet

In [ ]:
# TabNet Implementation
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.augmentations import ClassificationSMOTE

# TabNet with SMOTE augmentation
tabnet_model = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={"step_size":50, "gamma":0.9},
    mask_type='entmax'
)

# Train with SMOTE
tabnet_model.fit(
    X_train=X_train_scaled, y_train=y_train,
    eval_set=[(X_test_scaled, y_test)],
    max_epochs=100,
    patience=20,
    batch_size=256,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

# Evaluate
tabnet_preds = tabnet_model.predict(X_test_scaled)
tabnet_probs = tabnet_model.predict_proba(X_test_scaled)[:, 1]
tabnet_auc = roc_auc_score(y_test, tabnet_probs)
print(f"TabNet AUC: {tabnet_auc:.4f}")

# DNNs with advanced regularization 

In [ ]:
# Cell: Advanced DNN with BatchNorm
print("=== ADVANCED DNN WITH BATCHNORM ===")

class AdvancedDNN(nn.Module):
    def __init__(self, input_size, hidden_size=256):
        super(AdvancedDNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.BatchNorm1d(hidden_size // 4),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 4, 2)
        )
    
    def forward(self, x):
        return self.layers(x)

# Train and evaluate
print("Training Advanced DNN...")
advanced_model = AdvancedDNN(X_train_scaled.shape[1])
train_loader, test_loader, input_size = prepare_data_for_dl(X_train_scaled, X_test_scaled, y_train, y_test)

trained_advanced, train_losses, val_losses = train_model(advanced_model, train_loader, test_loader, epochs=30)
accuracy, auc, f1, preds, probs = evaluate_model(trained_advanced, test_loader, y_test)

print(f"\nAdvanced DNN Results:")
print(f"Accuracy: {accuracy:.4f}")
print(f"AUC: {auc:.4f}")
print(f"F1 Score: {f1:.4f}")

# Plot training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Advanced DNN - Training Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.hist(probs, bins=50, alpha=0.7)
plt.title('Advanced DNN - Prediction Probabilities')
plt.xlabel('Probability')
plt.ylabel('Count')

plt.subplot(1, 3, 3)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Advanced DNN - Confusion Matrix')
plt.tight_layout()
plt.show()

# feature selection with DL

In [ ]:
# Add missing imports for feature selection and ensemble methods
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import xgboost as xgb
from scipy.optimize import minimize

print("Imports added successfully!")

# Cell: Feature Selection + DL
print("=== FEATURE SELECTION + DEEP LEARNING ===")

# Select best features
selector = SelectKBest(score_func=f_classif, k=50)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

print(f"Selected {X_train_selected.shape[1]} features out of {X_train_scaled.shape[1]}")

# Train DL on selected features
train_loader_selected, test_loader_selected, input_size_selected = prepare_data_for_dl(
    X_train_selected, X_test_selected, y_train, y_test
)

# Train multiple models on selected features
models_selected = {
    'SimpleNN_Selected': SimpleNN(input_size_selected),
    'ResidualNN_Selected': ResidualNN(input_size_selected)
}

results_selected = {}

for name, model in models_selected.items():
    print(f"\nTraining {name}...")
    try:
        trained_model, train_losses, val_losses = train_model(
            model, train_loader_selected, test_loader_selected, epochs=25
        )
        
        accuracy, auc, f1, preds, probs = evaluate_model(trained_model, test_loader_selected, y_test)
        
        results_selected[name] = {
            'accuracy': accuracy,
            'auc': auc,
            'f1': f1,
            'train_losses': train_losses,
            'val_losses': val_losses
        }
        
        print(f"{name} Results:")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"AUC: {auc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        
    except Exception as e:
        print(f"Error with {name}: {e}")
        results_selected[name] = {'error': str(e)}

# Compare with full feature set
print("\n" + "="*60)
print("FEATURE SELECTION COMPARISON")
print("="*60)

comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Features': 'Selected (50)',
        'Accuracy': result.get('accuracy', np.nan),
        'AUC': result.get('auc', np.nan),
        'F1': result.get('f1', np.nan)
    }
    for name, result in results_selected.items() if 'error' not in result
])

print(comparison_df.to_string(index=False))

In [ ]:
# Cell: Ensemble Methods
print("=== ENSEMBLE METHODS ===")

# Traditional models
lr = LogisticRegression(random_state=42, max_iter=1000)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb_model = xgb.XGBClassifier(random_state=42)

# Train traditional models
print("Training traditional models...")
lr.fit(X_train_scaled, y_train)
rf.fit(X_train_scaled, y_train)
xgb_model.fit(X_train_scaled, y_train)

# Get predictions
lr_preds = lr.predict_proba(X_test_scaled)[:, 1]
rf_preds = rf.predict_proba(X_test_scaled)[:, 1]
xgb_preds = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Simple averaging ensemble
ensemble_preds = (lr_preds + rf_preds + xgb_preds) / 3
ensemble_auc = roc_auc_score(y_test, ensemble_preds)

print(f"\nIndividual Model AUCs:")
print(f"Logistic Regression: {roc_auc_score(y_test, lr_preds):.4f}")
print(f"Random Forest: {roc_auc_score(y_test, rf_preds):.4f}")
print(f"XGBoost: {roc_auc_score(y_test, xgb_preds):.4f}")
print(f"Ensemble (Average): {ensemble_auc:.4f}")

# Weighted ensemble (optimize weights)
def optimize_weights(weights, preds_list, y_true):
    ensemble_pred = np.zeros(len(y_true))
    for i, pred in enumerate(preds_list):
        ensemble_pred += weights[i] * pred
    return -roc_auc_score(y_true, ensemble_pred)

# Optimize weights
preds_list = [lr_preds, rf_preds, xgb_preds]
initial_weights = [1/3, 1/3, 1/3]
result = minimize(optimize_weights, initial_weights, args=(preds_list, y_test), 
                 bounds=[(0, 1)]*3, constraints={'type': 'eq', 'fun': lambda x: sum(x) - 1})

optimized_weights = result.x
optimized_ensemble_preds = sum(w * pred for w, pred in zip(optimized_weights, preds_list))
optimized_ensemble_auc = roc_auc_score(y_test, optimized_ensemble_preds)

print(f"\nOptimized Ensemble:")
print(f"Weights: {optimized_weights}")
print(f"AUC: {optimized_ensemble_auc:.4f}")

# Plot ensemble results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.hist(lr_preds, bins=50, alpha=0.7, label='LR')
plt.hist(rf_preds, bins=50, alpha=0.7, label='RF')
plt.hist(xgb_preds, bins=50, alpha=0.7, label='XGB')
plt.title('Individual Model Predictions')
plt.xlabel('Probability')
plt.ylabel('Count')
plt.legend()

plt.subplot(1, 3, 2)
plt.hist(ensemble_preds, bins=50, alpha=0.7, color='red')
plt.title('Simple Average Ensemble')
plt.xlabel('Probability')
plt.ylabel('Count')

plt.subplot(1, 3, 3)
plt.hist(optimized_ensemble_preds, bins=50, alpha=0.7, color='green')
plt.title('Optimized Weight Ensemble')
plt.xlabel('Probability')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

# plateau performance is likely a feature issue
- computation unlikely to improve much here, data is the issue. hence the following...

# advanced FE

In [ ]:
# Cell: Advanced Clinical Features
print("=== ADVANCED CLINICAL FEATURE ENGINEERING ===")

# Clinical thresholds based on literature
df_age_matched['aq_clinical_26'] = (df_age_matched['aq_total'] >= 26).astype(int)
df_age_matched['aq_clinical_32'] = (df_age_matched['aq_total'] >= 32).astype(int)
df_age_matched['eq_clinical_30'] = (df_age_matched['eq_total'] <= 30).astype(int)
df_age_matched['sqr_clinical_60'] = (df_age_matched['sqr_total'] >= 60).astype(int)

# Developmental age groups with specific features
df_age_matched['adolescent_13_17'] = ((df_age_matched['age'] >= 13) & (df_age_matched['age'] <= 17)).astype(int)
df_age_matched['young_adult_18_25'] = ((df_age_matched['age'] >= 18) & (df_age_matched['age'] <= 25)).astype(int)
df_age_matched['adult_26_40'] = ((df_age_matched['age'] >= 26) & (df_age_matched['age'] <= 40)).astype(int)

# Sex-specific clinical patterns
if 'sex_num' in df_age_matched.columns:
    df_age_matched['female_high_aq'] = ((df_age_matched['sex_num'] == 0) & (df_age_matched['aq_total'] >= 26)).astype(int)
    df_age_matched['male_high_aq'] = ((df_age_matched['sex_num'] == 1) & (df_age_matched['aq_total'] >= 26)).astype(int)
    df_age_matched['female_low_eq'] = ((df_age_matched['sex_num'] == 0) & (df_age_matched['eq_total'] <= 30)).astype(int)
    df_age_matched['male_low_eq'] = ((df_age_matched['sex_num'] == 1) & (df_age_matched['eq_total'] <= 30)).astype(int)

# Cognitive profile features
df_age_matched['cognitive_dysbalance'] = abs(df_age_matched['eq_total'] - df_age_matched['sqr_total'])
df_age_matched['social_cognitive_ratio'] = df_age_matched['eq_total'] / (df_age_matched['sqr_total'] + 1e-8)
df_age_matched['aq_eq_difference'] = df_age_matched['aq_total'] - df_age_matched['eq_total']

# Extreme value features (clinical significance)
for col in ['aq_total', 'eq_total', 'sqr_total']:
    q99 = df_age_matched[col].quantile(0.99)
    q01 = df_age_matched[col].quantile(0.01)
    df_age_matched[f'{col}_extreme_high'] = (df_age_matched[col] > q99).astype(int)
    df_age_matched[f'{col}_extreme_low'] = (df_age_matched[col] < q01).astype(int)

# Interaction features with clinical meaning
df_age_matched['age_aq_clinical'] = df_age_matched['age'] * df_age_matched['aq_clinical_26']
df_age_matched['age_eq_clinical'] = df_age_matched['age'] * df_age_matched['eq_clinical_30']
df_age_matched['sex_age_aq'] = df_age_matched['sex_num'] * df_age_matched['age'] * df_age_matched['aq_total']

print(f"After clinical features: {df_age_matched.shape}")   

# analyse data qual

In [ ]:
# Cell: Data Quality Analysis
print("=== DATA QUALITY ANALYSIS ===")

# Check for data leakage
print("Checking for data leakage...")
leakage_indicators = []
for col in X.columns:
    if any(term in col.lower() for term in ['diagnosis', 'clinical', 'assessment', 'evaluation']):
        leakage_indicators.append(col)

print(f"Potential leakage features: {leakage_indicators}")

# Check feature importance distribution
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 most important features:")
print(importance_df.head(10))

print(f"\nBottom 10 least important features:")
print(importance_df.tail(10))

# Check if any features are too predictive (potential leakage)
suspicious_features = importance_df[importance_df['importance'] > 0.1]
print(f"\nSuspiciously important features (>0.1):")
print(suspicious_features)

# Analyze prediction confidence
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]

print(f"\nPrediction confidence analysis:")
print(f"Mean prediction probability: {lr_probs.mean():.4f}")
print(f"Std prediction probability: {lr_probs.std():.4f}")
print(f"Min prediction probability: {lr_probs.min():.4f}")
print(f"Max prediction probability: {lr_probs.max():.4f}")

# Plot prediction distribution
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.hist(lr_probs, bins=50, alpha=0.7)
plt.title('Prediction Probability Distribution')
plt.xlabel('Probability')
plt.ylabel('Count')

plt.subplot(1, 3, 2)
plt.scatter(lr_probs, y_test, alpha=0.5)
plt.title('Predictions vs True Labels')
plt.xlabel('Predicted Probability')
plt.ylabel('True Label')

plt.subplot(1, 3, 3)
plt.barh(range(10), importance_df['importance'][:10])
plt.yticks(range(10), importance_df['feature'][:10])
plt.xlabel('Feature Importance')
plt.title('Top 10 Features')
plt.tight_layout()
plt.show()

# different target definitions

In [ ]:
# Cell: Alternative Target Definitions (FIXED - No Leakage)
print("=== ALTERNATIVE TARGET DEFINITIONS (FIXED) ===")

# Handle NaN values properly
print("Handling NaN values...")
missing_counts = df_age_matched.isnull().sum()
print(f"Missing values: {missing_counts.sum()}")

# Create clean dataframe
df_clean = df_age_matched.copy()

# Handle numeric and categorical columns separately
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns

# Fill numeric columns with mean
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].mean())

# Fill categorical columns with mode
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0] if len(df_clean[col].mode()) > 0 else 'Unknown'
        df_clean[col] = df_clean[col].fillna(mode_val)

print(f"After cleaning: {df_clean.shape}")

# 1. Continuous target (EQ score as regression - NOT AQ to avoid leakage)
print("1. Continuous EQ Score Prediction (avoiding AQ leakage):")
y_continuous = df_clean['eq_total']
X_continuous = df_clean.drop(['autism_target', 'aq_total', 'eq_total', 'sqr_total', 'spq_total'], axis=1)

# Remove any remaining leakage features
leakage_features = [col for col in X_continuous.columns if any(term in col.lower() for term in 
                   ['autism', 'risk_score', 'target', 'aq_', 'eq_', 'sqr_', 'spq_'])]
X_continuous = X_continuous.drop(columns=leakage_features)
X_continuous = X_continuous.select_dtypes(include=[np.number])
X_continuous = X_continuous.fillna(X_continuous.mean())

print(f"X_continuous shape: {X_continuous.shape}")
print(f"Features used: {list(X_continuous.columns)}")

# Split for regression
X_train_cont, X_test_cont, y_train_cont, y_test_cont = train_test_split(
    X_continuous, y_continuous, test_size=0.2, random_state=42
)

# Scale features
scaler_cont = StandardScaler()
X_train_cont_scaled = scaler_cont.fit_transform(X_train_cont)
X_test_cont_scaled = scaler_cont.transform(X_test_cont)

# Train regression model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

lr_reg = LinearRegression()
lr_reg.fit(X_train_cont_scaled, y_train_cont)
y_pred_cont = lr_reg.predict(X_test_cont_scaled)

r2 = r2_score(y_test_cont, y_pred_cont)
rmse = np.sqrt(mean_squared_error(y_test_cont, y_pred_cont))

print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

# 2. Severity levels based on EQ scores (avoiding AQ leakage)
print("\n2. Severity Level Classification (based on EQ scores):")
eq_scores = df_clean['eq_total']
y_severity = pd.cut(eq_scores, bins=[0, 20, 35, 80], labels=['low', 'medium', 'high'])
print(f"Severity distribution: {y_severity.value_counts()}")

# Prepare features for severity classification (avoiding questionnaire totals)
X_severity = df_clean.drop(['autism_target', 'aq_total', 'eq_total', 'sqr_total', 'spq_total'], axis=1)
leakage_features = [col for col in X_severity.columns if any(term in col.lower() for term in 
                   ['autism', 'risk_score', 'target', 'aq_', 'eq_', 'sqr_', 'spq_'])]
X_severity = X_severity.drop(columns=leakage_features)
X_severity = X_severity.select_dtypes(include=[np.number])
X_severity = X_severity.fillna(X_severity.mean())

# Remove rows with NaN in severity labels
valid_mask = ~y_severity.isna()
X_severity = X_severity[valid_mask]
y_severity = y_severity[valid_mask]

print(f"Valid samples for severity: {len(y_severity)}")
print(f"Severity distribution: {y_severity.value_counts()}")

# Only proceed if we have multiple classes
if len(y_severity.unique()) > 1:
    # Split for severity
    X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(
        X_severity, y_severity, test_size=0.2, random_state=42, stratify=y_severity
    )
    
    # Scale features
    scaler_sev = StandardScaler()
    X_train_sev_scaled = scaler_sev.fit_transform(X_train_sev)
    X_test_sev_scaled = scaler_sev.transform(X_test_sev)
    
    # Train severity classifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report
    
    rf_sev = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_sev.fit(X_train_sev_scaled, y_train_sev)
    y_pred_sev = rf_sev.predict(X_test_sev_scaled)
    
    sev_accuracy = accuracy_score(y_test_sev, y_pred_sev)
    print(f"Severity Classification Accuracy: {sev_accuracy:.4f}")
    print("\nSeverity Classification Report:")
    print(classification_report(y_test_sev, y_pred_sev))
else:
    print("Not enough classes for severity classification")
    sev_accuracy = 0.0

# 3. Different clinical thresholds (using EQ instead of AQ to avoid leakage)
print("\n3. Different Clinical Thresholds (EQ-based):")
thresholds = [15, 20, 25, 30, 35, 40, 45]
threshold_results = []

for threshold in thresholds:
    y_threshold = (df_clean['eq_total'] <= threshold).astype(int)  # Low EQ = potential issue
    print(f"Threshold {threshold}: {y_threshold.sum()} positive cases ({y_threshold.mean():.3f})")
    
    # Quick evaluation
    X_thresh = df_clean.drop(['autism_target', 'aq_total', 'eq_total', 'sqr_total', 'spq_total'], axis=1)
    leakage_features = [col for col in X_thresh.columns if any(term in col.lower() for term in 
                       ['autism', 'risk_score', 'target', 'aq_', 'eq_', 'sqr_', 'spq_'])]
    X_thresh = X_thresh.drop(columns=leakage_features)
    X_thresh = X_thresh.select_dtypes(include=[np.number])
    X_thresh = X_thresh.fillna(X_thresh.mean())
    
    # Only proceed if we have both classes
    if y_threshold.sum() > 0 and y_threshold.sum() < len(y_threshold):
        X_train_thresh, X_test_thresh, y_train_thresh, y_test_thresh = train_test_split(
            X_thresh, y_threshold, test_size=0.2, random_state=42, stratify=y_threshold
        )
        
        scaler_thresh = StandardScaler()
        X_train_thresh_scaled = scaler_thresh.fit_transform(X_train_thresh)
        X_test_thresh_scaled = scaler_thresh.transform(X_test_thresh)
        
        lr_thresh = LogisticRegression(random_state=42)
        lr_thresh.fit(X_train_thresh_scaled, y_train_thresh)
        y_pred_thresh = lr_thresh.predict_proba(X_test_thresh_scaled)[:, 1]
        auc_thresh = roc_auc_score(y_test_thresh, y_pred_thresh)
        
        threshold_results.append({'threshold': threshold, 'auc': auc_thresh})
        print(f"  AUC: {auc_thresh:.4f}")
    else:
        print(f"  Skipping - no variation in classes")

# Plot threshold results if we have any
if threshold_results:
    threshold_df = pd.DataFrame(threshold_results)
    plt.figure(figsize=(10, 6))
    plt.plot(threshold_df['threshold'], threshold_df['auc'], marker='o')
    plt.xlabel('EQ Threshold')
    plt.ylabel('AUC')
    plt.title('Performance vs EQ Threshold')
    plt.grid(True)
    plt.show()
else:
    print("No valid threshold results to plot")

# feature comninations analysis

In [ ]:
# Cell: Feature Combinations Analysis (FIXED - No Leakage)
print("=== FEATURE COMBINATIONS ANALYSIS (FIXED) ===")

# Use the cleaned data
df_clean = df_age_matched.copy()

# Handle NaN values properly
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns

df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].mean())
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0] if len(df_clean[col].mode()) > 0 else 'Unknown'
        df_clean[col] = df_clean[col].fillna(mode_val)

y_clean = df_clean['autism_target']
X_clean = df_clean.drop(['autism_target'], axis=1)

# Remove leakage features
leakage_features = [col for col in X_clean.columns if any(term in col.lower() for term in ['autism', 'risk_score', 'target'])]
X_clean = X_clean.drop(columns=leakage_features)
X_clean = X_clean.select_dtypes(include=[np.number])
X_clean = X_clean.fillna(X_clean.mean())

# Define feature groups (avoiding questionnaire totals)
aq_features = [col for col in X_clean.columns if 'aq_' in col and not col.endswith('_total')]
eq_features = [col for col in X_clean.columns if 'eq_' in col and not col.endswith('_total')]
sqr_features = [col for col in X_clean.columns if 'sqr_' in col and not col.endswith('_total')]
spq_features = [col for col in X_clean.columns if 'spq_' in col and not col.endswith('_total')]
demographic_features = [col for col in X_clean.columns if col in ['age', 'sex_num', 'is_stem_occupation']]
interaction_features = [col for col in X_clean.columns if any(term in col for term in ['_x_', '_ratio', '_interaction'])]

feature_groups = {
    'AQ_items_only': aq_features,
    'EQ_items_only': eq_features,
    'SQR_items_only': sqr_features,
    'SPQ_items_only': spq_features,
    'Demographic_only': demographic_features,
    'Interaction_only': interaction_features,
    'All_features': list(X_clean.columns)
}

print("Feature group sizes:")
for name, features in feature_groups.items():
    print(f"{name}: {len(features)} features")

# Test each feature group
feature_group_results = {}

for group_name, feature_list in feature_groups.items():
    if len(feature_list) == 0:
        continue
        
    print(f"\nTesting {group_name}...")
    
    # Select features
    X_group = X_clean[feature_list]
    
    # Split data
    X_train_group, X_test_group, y_train_group, y_test_group = train_test_split(
        X_group, y_clean, test_size=0.2, random_state=42, stratify=y_clean
    )
    
    # Scale features
    scaler_group = StandardScaler()
    X_train_group_scaled = scaler_group.fit_transform(X_train_group)
    X_test_group_scaled = scaler_group.transform(X_test_group)
    
    # Train model
    lr_group = LogisticRegression(random_state=42, max_iter=1000)
    lr_group.fit(X_train_group_scaled, y_train_group)
    y_pred_group = lr_group.predict_proba(X_test_group_scaled)[:, 1]
    auc_group = roc_auc_score(y_test_group, y_pred_group)
    
    feature_group_results[group_name] = {
        'n_features': len(feature_list),
        'auc': auc_group
    }
    
    print(f"  Features: {len(feature_list)}")
    print(f"  AUC: {auc_group:.4f}")

# Plot results
results_df = pd.DataFrame([
    {'Group': name, 'Features': result['n_features'], 'AUC': result['auc']}
    for name, result in feature_group_results.items()
]).sort_values('AUC', ascending=False)

print(f"\nFeature Group Performance:")
print(results_df.to_string(index=False))

# Plot feature group performance
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.bar(range(len(results_df)), results_df['AUC'])
plt.xticks(range(len(results_df)), results_df['Group'], rotation=45)
plt.ylabel('AUC')
plt.title('Performance by Feature Group')

plt.subplot(1, 2, 2)
plt.scatter(results_df['Features'], results_df['AUC'])
plt.xlabel('Number of Features')
plt.ylabel('AUC')
plt.title('Performance vs Feature Count')

for i, row in results_df.iterrows():
    plt.annotate(row['Group'], (row['Features'], row['AUC']), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Cell: Alternative Validation Strategies (FIXED)
print("=== ALTERNATIVE VALIDATION STRATEGIES ===")

# Use the cleaned data from your feature analysis
# Assuming you have X_clean and y_clean from the previous cell

# 1. Demographic stratified split (FIXED)
print("1. Demographic Stratified Split:")

# Create demographic strata with more reasonable grouping
df_age_matched['age_group'] = pd.cut(df_age_matched['age'], bins=[0, 25, 35, 50, 100], labels=['young', 'adult', 'middle', 'senior'])
df_age_matched['demographic_strata'] = df_age_matched['age_group'].astype(str) + '_' + df_age_matched['sex_num'].astype(str)
strata = df_age_matched['demographic_strata']

print(f"Strata distribution: {strata.value_counts()}")

# Use the cleaned data
X_clean = df_age_matched.drop(['autism_target'], axis=1)
leakage_features = [col for col in X_clean.columns if any(term in col.lower() for term in ['autism', 'risk_score', 'target'])]
X_clean = X_clean.drop(columns=leakage_features)
X_clean = X_clean.select_dtypes(include=[np.number])
X_clean = X_clean.fillna(X_clean.mean())

y_clean = df_age_matched['autism_target']

# Stratified split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(X_clean, y_clean):  # Use y_clean instead of strata
    X_train_strat, X_test_strat = X_clean.iloc[train_idx], X_clean.iloc[test_idx]
    y_train_strat, y_test_strat = y_clean.iloc[train_idx], y_clean.iloc[test_idx]

# Scale and train
scaler_strat = StandardScaler()
X_train_strat_scaled = scaler_strat.fit_transform(X_train_strat)
X_test_strat_scaled = scaler_strat.transform(X_test_strat)

lr_strat = LogisticRegression(random_state=42)
lr_strat.fit(X_train_strat_scaled, y_train_strat)
y_pred_strat = lr_strat.predict_proba(X_test_strat_scaled)[:, 1]
auc_strat = roc_auc_score(y_test_strat, y_pred_strat)

print(f"Demographic stratified AUC: {auc_strat:.4f}")

# 2. Cross-validation with different splits
print("\n2. Cross-Validation Results:")
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(LogisticRegression(random_state=42), X_train_scaled, y_train, 
                           cv=5, scoring='roc_auc')
print(f"5-fold CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# 3. Bootstrap validation
print("\n3. Bootstrap Validation:")
from sklearn.utils import resample

bootstrap_scores = []
for i in range(100):
    # Bootstrap sample
    X_boot, y_boot = resample(X_train_scaled, y_train, random_state=i)
    
    # Train and evaluate
    lr_boot = LogisticRegression(random_state=42)
    lr_boot.fit(X_boot, y_boot)
    y_pred_boot = lr_boot.predict_proba(X_test_scaled)[:, 1]
    auc_boot = roc_auc_score(y_test, y_pred_boot)
    bootstrap_scores.append(auc_boot)

print(f"Bootstrap AUC: {np.mean(bootstrap_scores):.4f} (+/- {np.std(bootstrap_scores) * 2:.4f})")

# Plot bootstrap distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(bootstrap_scores, bins=20, alpha=0.7)
plt.xlabel('AUC')
plt.ylabel('Count')
plt.title('Bootstrap AUC Distribution')

plt.subplot(1, 2, 2)
plt.boxplot([cv_scores, bootstrap_scores], labels=['CV', 'Bootstrap'])
plt.ylabel('AUC')
plt.title('Validation Method Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# Cell: Data Augmentation (FIXED - No imblearn)
print("=== DATA AUGMENTATION ===")

# Use the original cleaned data (X_train_scaled, y_train, X_test_scaled, y_test)
# These should already be cleaned from your previous cells

# 1. Simple oversampling (no SMOTE dependency)
print("1. Simple Oversampling:")
from sklearn.utils import resample

# Separate classes
X_train_0 = X_train_scaled[y_train == 0]
X_train_1 = X_train_scaled[y_train == 1]
y_train_0 = y_train[y_train == 0]
y_train_1 = y_train[y_train == 1]

# Oversample minority class (if any)
if len(X_train_1) < len(X_train_0):
    # Upsample minority class
    X_train_1_upsampled = resample(X_train_1, 
                                   n_samples=len(X_train_0), 
                                   random_state=42)
    y_train_1_upsampled = np.ones(len(X_train_0))
    
    # Combine
    X_train_oversampled = np.vstack([X_train_0, X_train_1_upsampled])
    y_train_oversampled = np.concatenate([y_train_0, y_train_1_upsampled])
else:
    # Upsample majority class
    X_train_0_upsampled = resample(X_train_0, 
                                   n_samples=len(X_train_1), 
                                   random_state=42)
    y_train_0_upsampled = np.zeros(len(X_train_1))
    
    # Combine
    X_train_oversampled = np.vstack([X_train_0_upsampled, X_train_1])
    y_train_oversampled = np.concatenate([y_train_0_upsampled, y_train_1])

print(f"Original train set: {len(y_train)} samples")
print(f"Oversampled train set: {len(y_train_oversampled)} samples")
print(f"Original class distribution: {np.bincount(y_train)}")
print(f"Oversampled class distribution: {np.bincount(y_train_oversampled)}")

# Train with oversampling
lr_oversampled = LogisticRegression(random_state=42)
lr_oversampled.fit(X_train_oversampled, y_train_oversampled)
y_pred_oversampled = lr_oversampled.predict_proba(X_test_scaled)[:, 1]
auc_oversampled = roc_auc_score(y_test, y_pred_oversampled)

print(f"Oversampled AUC: {auc_oversampled:.4f}")

# 2. Synthetic data generation
print("\n2. Synthetic Data Generation:")
from sklearn.mixture import GaussianMixture

# Fit GMM to generate synthetic samples
gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(X_train_scaled)

# Generate synthetic samples
n_synthetic = len(X_train_scaled) // 2
synthetic_samples = gmm.sample(n_synthetic)[0]

# Create synthetic labels (you'd need domain knowledge for this)
# For now, use the same distribution as original
synthetic_labels = np.random.choice([0, 1], size=n_synthetic, p=[0.5, 0.5])

# Combine with original data
X_train_synth = np.vstack([X_train_scaled, synthetic_samples])
y_train_synth = np.concatenate([y_train, synthetic_labels])

print(f"Synthetic train set: {len(y_train_synth)} samples")

# Train with synthetic data
lr_synth = LogisticRegression(random_state=42)
lr_synth.fit(X_train_synth, y_train_synth)
y_pred_synth = lr_synth.predict_proba(X_test_scaled)[:, 1]
auc_synth = roc_auc_score(y_test, y_pred_synth)

print(f"Synthetic data AUC: {auc_synth:.4f}")

# Compare all methods
methods_comparison = {
    'Original': roc_auc_score(y_test, LogisticRegression(random_state=42).fit(X_train_scaled, y_train).predict_proba(X_test_scaled)[:, 1]),
    'Oversampled': auc_oversampled,
    'Synthetic': auc_synth
}

print(f"\nMethod Comparison:")
for method, auc in methods_comparison.items():
    print(f"{method}: {auc:.4f}")

# Plot comparison
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(methods_comparison.keys(), methods_comparison.values())
plt.ylabel('AUC')
plt.title('Data Augmentation Methods')

plt.subplot(1, 2, 2)
plt.hist(y_pred_oversampled, bins=50, alpha=0.7, label='Oversampled')
plt.hist(y_pred_synth, bins=50, alpha=0.7, label='Synthetic')
plt.xlabel('Prediction Probability')
plt.ylabel('Count')
plt.title('Prediction Distributions')
plt.legend()
plt.tight_layout()
plt.show()